# CropGrowth Agent
Tell it what to map in plain language. It picks the right product, checks the place names, asks before long runs,
processes satellite NDVI (Harmonized Landsat Sentinel-2 by default) and saves rice growth-stage maps to Google Drive
or Cloud Storage.

* **Municipal / barangay** → the **current** growth stage + days since the last clear observation + hectares per stage.
* **Provincial / regional / national** → one stage map per province per **month** or **half-month**.

**Setup once**
1. Add your Gemini API key as a Colab secret named `GEMINI_API_KEY` (🔑 icon, left sidebar).
   Optional: add `OPENROUTER_API_KEY` too. When Gemini is busy (rate limit / overload) the agent then
   continues on Qwen via OpenRouter.
2. The repository is private: add a read-only GitHub token as `GITHUB_TOKEN`, or copy the repo folder to Drive and set `REPO_DIR` to it.
3. Put the boundary files on Drive: provinces (with planting month, and a region column for regional runs) and
   municipalities / barangays.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# the repository (folder that contains cropgrowth_agent/):
#  - a copy on Drive: set REPO_DIR to it, or
#  - clone it here: private repo -> add a read-only GitHub token as the Colab secret GITHUB_TOKEN
import os, subprocess
REPO_DIR = '/content/ai-cropgrowth'
REPO_URL = 'github.com/Leads-DigiSaka-System/ai-cropgrowth.git'
BRANCH   = 'main'
if not os.path.isdir(os.path.join(REPO_DIR, 'cropgrowth_agent')):
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, f'https://x-access-token:{tok}@{REPO_URL}', REPO_DIR], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'], check=True)  # don't keep the token
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=False)
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
import os, sys
from google.colab import userdata

PROVINCES  = '/content/drive/MyDrive/AI-CropGrowth/vector/boundary_province_philippines.gpkg'
AOIS       = '/content/drive/MyDrive/AI-CropGrowth/vector/boundary_municipal_philippines.gpkg'
OUTPUT     = 'gdrive'                                         # 'gdrive' | 'gcs' | 'both' | 'local'
DRIVE_OUT  = '/content/drive/MyDrive/AI-CropGrowth/outputs'

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
try:
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')   # optional: Qwen fallback
except Exception:
    pass
sys.path.insert(0, REPO_DIR)

from cropgrowth_agent import CropGrowthAgent, RunConfig
cfg = RunConfig(vector_path=PROVINCES, prov_col='Pro_Name', region_col='Reg_Name', plant_mo_col='Semester_1',
                aoi_path=AOIS, aoi_name_col='Mun_Name',
                year=2025, cog_label='dry2026', output_prefix='products/growth_stage/2026/dry',
                output_target=OUTPUT, drive_root=DRIVE_OUT,
                tile_cache_dir='/content/drive/MyDrive/AI-CropGrowth/tile_cache')
gcs_client = None                    # for OUTPUT 'gcs' / 'both': a google.cloud.storage.Client (see run_growth_stages.ipynb)
agent = CropGrowthAgent(config=cfg, gcs_client=gcs_client)

## Ask it

In [ ]:
print(agent.run("Which municipalities can you map in Nueva Ecija? Just the first few."))

In [ ]:
print(agent.run("What is the current growth stage of rice in Science City of Muñoz? Save the map to Google Drive."))

Follow-ups keep context, for example:
- `"Make last month's maps for Region III, semimonthly."` (it shows the plan and asks before running)
- `"Which provinces failed? Retry them."`
- `"Save to both Drive and Cloud Storage from now on."`
- `"How is the harvest date detected?"` (answered from the repo docs)

In [ ]:
agent.chat()      # interactive (empty line to stop)

## Look at the result

In [ ]:
import pandas as pd
rec = agent.runner.last_recent
if rec and rec['success']:
    s = rec['success'][0]
    print(s['area'], s['as_of'], '->', s['uri'])
    display(pd.DataFrame({'hectares': s['hectares'], 'share_%': s['share_pct']}))

## Without the LLM (same pipeline, direct call)
Useful for scheduled runs or when you already know the areas and settings.

In [ ]:
runner = agent.runner
print(runner.plan_periodic('regional', ['REGION III'])['tiles_to_process'], 'tiles')
# res = runner.run_periodic('regional', ['REGION III'])                       # this month's / last month's maps
# res = runner.run_recent(['SCIENCE CITY OF MUÑOZ'])                         # current stage